# Upstage API 실습

Upstage Document AI로 문서를 구조화하고, OpenAI Responses API로 요약·분류하는 실습 노트북입니다.

> 파일명은 요청에 맞춰 `function_calling.ipynb`로 만들었으며, 노션 페이지의 Upstage API 내용을 실행 가능한 셀로 재구성했습니다.


## 학습 목표

1. Upstage Document OCR과 Document Parse의 용도를 구분한다.
2. 이미지·PDF를 Upstage Document AI로 분석한다.
3. 파싱 결과를 Markdown/HTML/Text 파일로 저장한다.
4. OpenAI Responses API로 문서를 요약하고 Structured Outputs로 문서 종류를 분류한다.


## 0. 준비 사항

- Upstage Console에서 `UPSTAGE_API_KEY`를 발급한다.
- OpenAI API를 사용하는 셀은 `OPENAI_API_KEY`도 필요하다.
- 키는 코드에 직접 쓰지 말고 프로젝트 최상위의 `.env` 파일에 저장한다.

```text
UPSTAGE_API_KEY=up_발급받은_키
OPENAI_API_KEY=sk-발급받은_키
```


In [ ]:
%pip install -q requests python-dotenv beautifulsoup4 markdownify openai pydantic


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(override=True)
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not UPSTAGE_API_KEY:
    print("⚠️ UPSTAGE_API_KEY가 없습니다. .env 파일을 확인하세요.")
else:
    print("✅ Upstage API 키를 불러왔습니다.")


## 1. Upstage Document AI

| 구분 | Document OCR | Document Parse |
| --- | --- | --- |
| 목적 | 이미지·스캔 문서에서 글자 인식 | 문서의 구조와 읽기 순서까지 분석 |
| 추천 입력 | 영수증, 사진, 간단한 스캔 | PDF, 보고서, 표·그림·다단 문서 |
| 모델 별칭 | `ocr` | `document-parse` |
| 주요 결과 | 텍스트와 OCR 관련 정보 | `content.markdown`, `content.html`, `content.text`, `elements` |

강의와 일반 프로젝트에서는 특정 스냅샷 이름보다 안정 버전으로 연결되는 모델 별칭 `ocr`, `document-parse` 사용을 권장합니다.


### 1-1. OCR: 이미지에서 텍스트 추출

`IMAGE_FILE`을 분석할 실제 이미지 파일명으로 바꾸고 실행하세요.


In [ ]:
import requests

IMAGE_FILE = "abc.png"  # 분석할 이미지 파일명
url = "https://api.upstage.ai/v1/document-digitization"
headers = {"Authorization": f"Bearer {UPSTAGE_API_KEY}"}

with open(IMAGE_FILE, "rb") as image_file:
    response = requests.post(
        url,
        headers=headers,
        files={"document": image_file},
        data={"model": "ocr"},
        timeout=120,
    )

response.raise_for_status()
ocr_result = response.json()
print(ocr_result)

### 1-2. Document Parse: PDF를 LLM 친화적인 Markdown으로 변환

- `ocr="auto"`: 이미지형 문서에는 OCR을 적용하고, 디지털 PDF에는 필요한 분석을 수행합니다.
- 스캔 PDF에서 글자가 누락되면 `ocr="force"`를 검토합니다.
- 대용량 문서는 비동기 Document Parse API를 사용합니다.


In [ ]:
PDF_FILE = Path(".\\data\\sample.pdf")  # 분석할 PDF 파일명

with PDF_FILE.open("rb") as pdf_file:
    response = requests.post(
        url,
        headers=headers,
        files={"document": pdf_file},
        data={
            "model": "document-parse",
            "ocr": "auto",
        },
        timeout=120,
    )

response.raise_for_status()
parse_result = response.json()

# 결과 필드가 항상 존재한다고 가정하지 말고 안전하게 꺼냅니다.
content = parse_result.get("content", {})
markdown = content.get("markdown", "")
html = content.get("html", "")
plain_text = content.get("text", "")

print(html[:1000])


### 1-3. 파싱 결과 저장하기

HTML 원본, LLM/RAG에 활용하기 좋은 Markdown, 순수 텍스트를 각각 저장합니다.


In [ ]:
from bs4 import BeautifulSoup
from markdownify import markdownify as md

# API가 Markdown을 제공하지 않은 경우 HTML을 Markdown으로 변환합니다.
if not markdown and html:
    markdown = md(html, heading_style="ATX")

if html:
    Path("sample_parsed.html").write_text(html, encoding="utf-8")

Path("sample_parsed.md").write_text(markdown, encoding="utf-8")

if not plain_text and html:
    plain_text = BeautifulSoup(html, "html.parser").get_text("\n", strip=True)
Path("sample_parsed.txt").write_text(plain_text, encoding="utf-8")

print("저장 완료: sample_parsed.html / sample_parsed.md / sample_parsed.txt")
print(markdown[:1000])


## 2. OpenAI Responses API

새 프로젝트에서는 이전 Chat Completions 방식보다 **Responses API**를 기본 인터페이스로 사용하는 것을 권장합니다. 텍스트·이미지·파일 입력, Structured Outputs, 함수 호출과 도구 연동을 한 흐름으로 구현할 수 있습니다.

### 이미지 입력 예제

사진의 간단한 내용 설명이나 글자 읽기에는 OpenAI 이미지 입력을 사용할 수 있습니다. 표·문서 구조·좌표 정보가 중요하면 Upstage Document Parse/OCR이 더 적합합니다.


In [ ]:
import base64
from openai import OpenAI

client = OpenAI()
IMAGE_FILE = ".\\images\\sample.png"

with open(IMAGE_FILE, "rb") as image_file:
    image_b64 = base64.b64encode(image_file.read()).decode("utf-8")

response = client.responses.create(
    model="gpt-5-mini",
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "이미지의 핵심 내용과 읽을 수 있는 글자를 알려줘."},
            {
                "type": "input_image",
                "image_url": f"data:image/png;base64,{image_b64}",
            },
        ],
    }],
)

print(response.output_text)


## 3. Upstage Parse + OpenAI 요약 통합 예제

PDF를 Upstage로 구조화한 Markdown으로 바꾼 뒤, OpenAI가 핵심 내용을 요약합니다. 긴 문서는 한 번에 보내기보다 **청크별 요약 → 전체 요약** 방식이 더 안정적입니다.


In [ ]:
def parse_document_to_markdown(file_path: str) -> str:
    """PDF를 Upstage Document Parse로 분석하고 Markdown을 반환한다."""
    with open(file_path, "rb") as file:
        response = requests.post(
            "https://api.upstage.ai/v1/document-digitization",
            headers={"Authorization": f"Bearer {UPSTAGE_API_KEY}"},
            files={"document": file},
            data={"model": "document-parse", "ocr": "auto"},
            timeout=120,
        )
    response.raise_for_status()

    parsed_content = response.json().get("content", {})
    return parsed_content.get("html", "")

markdown = parse_document_to_markdown(".\\data\\sample.pdf")
Path("sample_parsed.md").write_text(markdown, encoding="utf-8")

response = client.responses.create(
    model="gpt-5-mini",
    instructions=(
        "다음 문서를 한국어로 요약하라. "
        "1) 핵심 요약 3개, 2) 중요한 수치, 3) 다음 행동 제안을 Markdown으로 작성하라."
    ),
    input=markdown,
)

summary = response.output_text
Path("sample_summary.md").write_text(summary, encoding="utf-8")
print(summary)


## 4. Structured Outputs로 문서 종류 인식하기

프롬프트로 JSON 모양의 문자열을 요구하는 것보다 `responses.parse()`와 Pydantic 모델을 사용하면 결과 형식을 검증된 객체로 받을 수 있습니다.


In [ ]:
from pydantic import BaseModel, Field

class DocumentInfo(BaseModel):
    document_type: str = Field(
        description="invoice, receipt, contract, cv, others 중 하나"
    )
    confidence: int = Field(ge=0, le=100)
    reason: str

response = client.responses.parse(
    model="gpt-5-mini",
    input=[
        {
            "role": "system",
            "content": (
                "문서 내용을 읽고 document_type을 "
                "invoice, receipt, contract, cv, others 중 하나로 분류하라."
            ),
        },
        {
            "role": "user",
            "content": "상호명: 코덱스마트\n총 결제금액: 18,500원\n카드 승인번호: 123456",
        },
    ],
    text_format=DocumentInfo,
)

info = response.output_parsed
print(info.document_type, info.confidence, info.reason)


## 5. 미니 과제

`sample.pdf`를 Document Parse로 변환하고 OpenAI API로 아래 형식의 결과를 생성하세요.

```text
제목:
핵심 내용 3개:
중요 수치:
확인이 필요한 내용:
```

완성 후에는 다음 3개 파일을 비교하세요.

- 원본: `sample.pdf`
- 중간 결과: `sample_parsed.md`
- 최종 결과: `sample_summary.md`


In [ ]:
# 미니 과제 시작 코드
# 1) sample.pdf를 파싱해 sample_parsed.md로 저장합니다.
# 2) 아래 형식의 요약 프롬프트를 작성합니다.
# 3) 모델 요약 결과를 sample_summary.md로 저장합니다.

assignment_format = """제목:
핵심 내용 3개:
중요 수치:
확인이 필요한 내용:
"""

# 여기에 코드를 작성하세요.
